
# Core Notebook 5 · Core-Only End-to-End Workflow

Tie the earlier pieces together by valuing a simple fixed-rate loan using only `finstack.core`. We will configure rounding, build schedules, assemble cashflows, construct curves/FX, and compute PV + IRR.


## Imports

In [24]:

from datetime import date

from finstack.core import currency, money, config, cashflow, dates
from finstack.core.dates.daycount import DayCount, DayCountContext, DayCountContextState
from finstack.core.dates.schedule import Frequency, ScheduleBuilder, StubKind
from finstack.core.market_data import term_structures, fx, context



## Configuration & instrument inputs

In [25]:

instrument_currency = currency.Currency("USD")
cfg = config.FinstackConfig()
cfg.set_rounding_mode("Bankers")
cfg.set_output_scale(instrument_currency, 2)

principal = money.Money(1_000_000, instrument_currency)
annual_coupon = 0.045
issue_date = date(2025, 1, 15)
maturity_date = date(2027, 1, 15)
print("Instrument:", principal, "coupon=", annual_coupon)


Instrument: USD 1000000.00 coupon= 0.045


## Schedule & day-count setup

In [26]:

us_calendar = dates.get_calendar("usny")
frequency = Frequency.from_months(6)
builder = ScheduleBuilder.new(issue_date, maturity_date)
coupon_schedule = (
    builder
    .frequency(frequency)
    .stub_rule(StubKind.SHORT_FRONT)
    .adjust_with(dates.BusinessDayConvention.MODIFIED_FOLLOWING, us_calendar)
    .end_of_month(False)
    .build()
)

dc = DayCount.ACT_ACT
dc_ctx = DayCountContext(calendar=us_calendar, frequency=frequency)
print("Schedule dates:")
for dt in coupon_schedule.dates:
    print(" ·", dt)


Schedule dates:
 · 2025-01-15
 · 2025-07-15
 · 2026-01-15
 · 2026-07-15
 · 2027-01-15


## Cashflow construction

In [27]:
coupon_cashflows = []
schedule_dates = coupon_schedule.dates
for start, end in zip(schedule_dates[:-1], schedule_dates[1:]):
    accrual = dc.year_fraction(start, end, dc_ctx)
    coupon_amount = principal.amount * annual_coupon * accrual
    cf = cashflow.CashFlow(
        end,
        money.Money(coupon_amount, instrument_currency),
        cashflow.CFKind.FIXED,
        accrual_factor=accrual,
        reset_date=start,
    )
    cf.validate()
    coupon_cashflows.append(cf)
maturity_cf = cashflow.CashFlow(schedule_dates[-1], principal, cashflow.CFKind.NOTIONAL)
cashflows = coupon_cashflows + [maturity_cf]
print("Total cashflows:", len(cashflows))


Total cashflows: 5


In [28]:
cashflows

[CashFlow(kind=fixed, date=2025-07-15, amount=USD 22315.07, reset_date=2025-01-15, accrual_factor=0.4958904109589041),
 CashFlow(kind=fixed, date=2026-01-15, amount=USD 22684.93, reset_date=2025-07-15, accrual_factor=0.5041095890410958),
 CashFlow(kind=fixed, date=2026-07-15, amount=USD 22315.07, reset_date=2026-01-15, accrual_factor=0.4958904109589041),
 CashFlow(kind=fixed, date=2027-01-15, amount=USD 22684.93, reset_date=2026-07-15, accrual_factor=0.5041095890410958),
 CashFlow(kind=notional, date=2027-01-15, amount=USD 1000000.00, reset_date=None, accrual_factor=0)]

## Curve & FX setup

In [29]:

base_date = schedule_dates[0]
curve_points = [
    (0.0, 1.0),
    (0.5, 0.9975),
    (1.0, 0.9940),
    (1.5, 0.9880),
    (2.0, 0.9825),
]
discount_curve = term_structures.DiscountCurve(
    id="USD-OIS",
    base_date=base_date,
    knots=curve_points,
    day_count=dc,
    interp="linear",
    extrapolation="flat_zero",
)

eur = currency.Currency("EUR")
fx_matrix = fx.FxMatrix()
fx_matrix.set_quote(instrument_currency, eur, 0.94)

market_ctx = context.MarketContext()
market_ctx.insert_discount(discount_curve)
market_ctx.insert_fx(fx_matrix)
print("Market context counts:", market_ctx.count_by_type())


Market context counts: {'Discount': 1}


## Pricing & IRR

In [30]:

def present_value(cashflow_list, curve):
    total = money.Money.zero(instrument_currency)
    for cf in cashflow_list:
        df = curve.df_on_date(cf.date)
        discounted = money.Money(cf.amount.amount * df, instrument_currency)
        total = total.checked_add(discounted)
    return total

pv = present_value(cashflows, discount_curve)
print("Discounted PV:", pv.format())

projected_flows = [(issue_date, -principal.amount)] + [
    (cf.date, cf.amount.amount) for cf in cashflows
]
workflow_irr = cashflow.xirr(projected_flows)
print(f"XIRR over constructed schedule: {workflow_irr:.6f}")


Discounted PV: USD 1071644.89
XIRR over constructed schedule: 0.045506


In [31]:
projected_flows

[(datetime.date(2025, 1, 15), -1000000.0),
 (datetime.date(2025, 7, 15), 22315.07),
 (datetime.date(2026, 1, 15), 22684.93),
 (datetime.date(2026, 7, 15), 22315.07),
 (datetime.date(2027, 1, 15), 22684.93),
 (datetime.date(2027, 1, 15), 1000000.0)]

## Serialization touchpoints

In [32]:

ctx_state = dc_ctx.to_state()
ctx_json = ctx_state.to_json()
restored_ctx = DayCountContextState.from_json(ctx_json).to_context()
print("Day-count context JSON:", ctx_json)
print("Restored frequency months:", restored_ctx.frequency.months)

curve_snapshot = discount_curve.points
print("Curve knots ready for persistence:", curve_snapshot)


Day-count context JSON: {
  "calendar_id": "usny",
  "frequency": {
    "Months": 6
  },
  "bus_basis": null
}
Restored frequency months: 6
Curve knots ready for persistence: [(0.0, 1.0), (0.5, 0.9975), (1.0, 0.994), (1.5, 0.988), (2.0, 0.9825)]



## Checklist · best practices

- Configure rounding/scales up front and pass `FinstackConfig` around instead of magic numbers
- Persist schedule parameters, day-count context states, and curve knots/IDs so the workflow can be replayed later
- Discount cashflows with the canonical `DiscountCurve`; use `FxMatrix` when collapsing to a base currency
- Use `cashflow.xirr` / solver utilities as sanity checks on PV results
